[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aelkharrim/FDM5-3P/blob/main/notebooks/08_incertitudes.ipynb)

# Python pour la physique — 8. Incertitudes de mesure

> 💡 **Astuce**
>
> **Rappel** : exécutez les cellules dans l'ordre avec **Maj + Entrée**. Les solutions des exercices sont masquées dans des cellules « Solution » : essayez d'abord par vous-même !


Une mesure sans incertitude n'a aucun sens en physique : dire que $g = 9{,}7\ \text{m/s}^2$ ne permet pas de savoir si c'est compatible avec $9{,}81\ \text{m/s}^2$. Dans ce carnet, nous voyons comment **calculer**, **propager** et **utiliser** les incertitudes avec Python.

In [ ]:
# Cellule de préparation (facultative) : exécutez-la une fois au début.
# Sur Google Colab, NumPy, SciPy, Matplotlib et SymPy sont déjà installés.
import sys
EN_COLAB = 'google.colab' in sys.modules
print('Exécution dans Google Colab' if EN_COLAB else 'Exécution en local')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sympy as sym
from scipy import stats, optimize

# Incertitude de type A : mesures répétées

Quand on répète $N$ fois la même mesure, on estime :

- la **moyenne** $\bar{x} = \dfrac{1}{N}\sum_i x_i$ ;
- l'**écart-type expérimental** $s = \sqrt{\dfrac{1}{N-1}\sum_i (x_i - \bar{x})^2}$ ;
- l'**incertitude-type sur la moyenne** $u(\bar{x}) = \dfrac{s}{\sqrt{N}}$.

Voici 10 mesures de la période d'un pendule (en s) :

In [ ]:
T = np.array([2.01, 1.98, 2.03, 2.00, 1.99, 2.02, 2.05, 1.97, 2.00, 2.01])

N = T.size
moyenne = T.mean()
s = T.std(ddof=1)          # ddof=1 : division par N-1 (écart-type expérimental)
u_moy = s / np.sqrt(N)

print(f"N = {N}")
print(f"moyenne           = {moyenne:.4f} s")
print(f"écart-type exp. s = {s:.4f} s")
print(f"u(moyenne)        = {u_moy:.4f} s")

> ⚠️ **Attention**
>
> Par défaut, `np.std` divise par $N$ (écart-type de la **population**). Pour des mesures, utilisez `ddof=1` (division par $N-1$). Avec pandas, `Series.std()` utilise déjà `ddof=1`.

## Écrire correctement un résultat

On arrondit l'incertitude à **1 ou 2 chiffres significatifs**, puis on arrondit la valeur au même rang. Écrivons une petite fonction qui le fait :

In [ ]:
def formate(valeur, incertitude, unite="", chiffres=2):
    """Formate 'valeur ± incertitude' avec le bon nombre de chiffres."""
    if incertitude == 0:
        return f"{valeur} {unite}"
    ordre = int(np.floor(np.log10(abs(incertitude))))
    decimales = max(0, chiffres - 1 - ordre)
    return f"({valeur:.{decimales}f} ± {incertitude:.{decimales}f}) {unite}".strip()

print(formate(moyenne, u_moy, "s"))
print(formate(9.7512, 0.0834, "m/s²"))
print(formate(1234.5, 12.0, "N", chiffres=1))

## Intervalle de confiance et coefficient de Student

Avec un petit nombre de mesures, la moyenne suit une loi de **Student** à $N-1$ degrés de liberté. L'**incertitude élargie** à 95 % est $U = t \cdot u(\bar{x})$, où $t$ est le coefficient de Student :

In [ ]:
niveau = 0.95
t_student = stats.t.ppf(1 - (1 - niveau) / 2, df=N - 1)
U = t_student * u_moy
print(f"t(95 %, {N-1} ddl) = {t_student:.3f}   (≈ 2 pour la loi normale)")
print(f"T = {formate(moyenne, U, 's')}  à 95 %")

`scipy.stats` fait le calcul directement :

In [ ]:
stats.t.interval(0.95, df=N - 1, loc=moyenne, scale=u_moy)

> **Exercice**
>
> On mesure 5 fois une résistance (en Ω) : `[98.7, 101.2, 99.5, 100.8, 99.1]`. Donnez la moyenne, l'écart-type expérimental, l'incertitude-type sur la moyenne, puis le résultat avec une incertitude élargie à 95 % (coefficient de Student pour $N-1 = 4$ degrés de liberté).

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
R = np.array([98.7, 101.2, 99.5, 100.8, 99.1])
u = R.std(ddof=1) / np.sqrt(R.size)
t95 = stats.t.ppf(0.975, df=R.size - 1)
print(f"moyenne = {R.mean():.2f} Ω, s = {R.std(ddof=1):.2f} Ω, u = {u:.2f} Ω")
print("t =", round(t95, 2))
print("R =", formate(R.mean(), t95 * u, "Ω"), "à 95 %")

# Incertitude de type B : l'instrument

Quand on ne fait qu'**une seule mesure**, ou quand l'incertitude vient de l'instrument, on l'évalue à partir des données du constructeur ou de la lecture. Si la grandeur est connue à $\pm a$ avec une probabilité uniforme (distribution rectangulaire), l'incertitude-type est

$$u = \frac{a}{\sqrt{3}}$$

Exemples :

In [ ]:
# Règle graduée au mm : lecture à ± 0,5 mm près (demi-graduation)
a_regle = 0.5e-3
print("u(règle)   =", a_regle / np.sqrt(3), "m")

# Chronomètre au centième, avec incertitude de réaction de ± 0,1 s
a_chrono = 0.1
print("u(chrono)  =", a_chrono / np.sqrt(3), "s")

Lorsque plusieurs sources indépendantes interviennent (type A + type B), on les combine **en quadrature** :

$$u_c = \sqrt{u_A^2 + u_B^2}$$

In [ ]:
u_A = u_moy
u_B = 0.1 / np.sqrt(3)
u_c = np.hypot(u_A, u_B)          # √(uA² + uB²)
print(f"uA = {u_A:.4f} s, uB = {u_B:.4f} s  ->  u_c = {u_c:.4f} s")

# Propagation des incertitudes

On mesure des grandeurs $x_1, x_2, \dots$ avec leurs incertitudes, et on calcule $y = f(x_1, x_2, \dots)$. Pour des grandeurs **indépendantes**, l'incertitude-type sur $y$ est :

$$u(y) = \sqrt{\sum_i \left(\frac{\partial f}{\partial x_i}\right)^2 u(x_i)^2}$$

## Méthode 1 : formule, avec SymPy

SymPy calcule les dérivées partielles pour nous. Exemple : $g = \dfrac{4\pi^2 L}{T^2}$, avec $L = (1{,}000 \pm 0{,}002)$ m et $T = (2{,}006 \pm 0{,}008)$ s.

In [ ]:
L_s, T_s = sym.symbols('L T', positive=True)
uL_s, uT_s = sym.symbols('u_L u_T', positive=True)

g_s = 4 * sym.pi**2 * L_s / T_s**2
u_g_s = sym.sqrt((g_s.diff(L_s) * uL_s)**2 + (g_s.diff(T_s) * uT_s)**2)
sym.simplify(u_g_s)

On remplace ensuite par les valeurs numériques :

In [ ]:
valeurs = {L_s: 1.000, T_s: 2.006, uL_s: 0.002, uT_s: 0.008}
g_val = float(g_s.subs(valeurs))
u_g_val = float(u_g_s.subs(valeurs))
print("g =", formate(g_val, u_g_val, "m/s²"))

On vérifie la formule habituelle pour un produit de puissances : $\left(\dfrac{u_g}{g}\right)^2 = \left(\dfrac{u_L}{L}\right)^2 + \left(2\dfrac{u_T}{T}\right)^2$.

In [ ]:
u_rel = np.sqrt((0.002/1.000)**2 + (2*0.008/2.006)**2)
g_val * u_rel, u_g_val

## Méthode 2 : simulation de Monte-Carlo

Quand la formule est complexe, ou que les incertitudes sont grandes, on peut **simuler** : on tire au hasard un grand nombre de valeurs de chaque grandeur d'entrée selon sa distribution, on calcule $y$ pour chaque tirage, et on regarde la dispersion des résultats.

In [ ]:
rng = np.random.default_rng(0)
n = 100_000
L_mc = rng.normal(1.000, 0.002, n)
T_mc = rng.normal(2.006, 0.008, n)
g_mc = 4 * np.pi**2 * L_mc / T_mc**2

print(f"Monte-Carlo : g = {g_mc.mean():.3f} ± {g_mc.std(ddof=1):.3f} m/s²")
print(f"Formule     : g = {g_val:.3f} ± {u_g_val:.3f} m/s²")

plt.hist(g_mc, bins=80, density=True, edgecolor='none')
plt.xlabel("g (m/s²)"); plt.ylabel("densité")
plt.title("Distribution de g par Monte-Carlo")
plt.show()

Les deux méthodes donnent le même résultat ici. La méthode de Monte-Carlo est plus générale, et elle permet de **visualiser** la distribution du résultat.


> **Exercice**
>
> Une résistance est mesurée à partir de $U = (5{,}00 \pm 0{,}05)$ V et $I = (0{,}250 \pm 0{,}004)$ A. Calculez $R = U/I$ et son incertitude-type, avec **les deux méthodes** (SymPy puis Monte-Carlo).

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
U_s, I_s, uU_s, uI_s = sym.symbols('U I u_U u_I', positive=True)
R_s = U_s / I_s
uR_s = sym.sqrt((R_s.diff(U_s) * uU_s)**2 + (R_s.diff(I_s) * uI_s)**2)
vals = {U_s: 5.00, I_s: 0.250, uU_s: 0.05, uI_s: 0.004}
print("formule     :", float(R_s.subs(vals)), "±", float(uR_s.subs(vals)))

rng = np.random.default_rng(1)
Umc = rng.normal(5.00, 0.05, 200_000)
Imc = rng.normal(0.250, 0.004, 200_000)
Rmc = Umc / Imc
print("Monte-Carlo :", Rmc.mean(), "±", Rmc.std(ddof=1))

# Comparer une mesure à une valeur de référence

Pour savoir si une mesure $x \pm u$ est **compatible** avec une valeur de référence $x_\text{ref}$, on calcule l'**écart normalisé** (ou *z-score*) :

$$E_n = \frac{|x - x_\text{ref}|}{u}$$

La convention usuelle : si $E_n < 2$, la mesure est compatible avec la référence (au niveau de confiance de 95 %). Si $E_n > 3$, l'écart est difficilement attribuable au hasard.

In [ ]:
g_ref = 9.81
En = abs(g_val - g_ref) / u_g_val
print(f"écart normalisé E_n = {En:.2f}")
print("compatible" if En < 2 else "non compatible", "avec", g_ref, "m/s²")

Pour comparer **deux mesures** indépendantes $x_1 \pm u_1$ et $x_2 \pm u_2$ :

$$E_n = \frac{|x_1 - x_2|}{\sqrt{u_1^2 + u_2^2}}$$

In [ ]:
def ecart_normalise(x1, u1, x2, u2):
    return abs(x1 - x2) / np.hypot(u1, u2)

ecart_normalise(9.78, 0.05, 9.85, 0.06)

# Régression linéaire avec incertitudes

On mesure $y$ en fonction de $x$, et on veut ajuster une droite $y = ax + b$ en tenant compte des incertitudes sur $y$. Exemple : l'allongement d'un ressort en fonction de la masse suspendue ($F = mg = k\,\Delta\ell$).

In [ ]:
rng = np.random.default_rng(4)
masse = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40])   # kg
u_allong = 0.002                                                       # m
k_vrai = 20.0
allong = masse * 9.81 / k_vrai + rng.normal(0, u_allong, masse.size)

donnees = pd.DataFrame({'m_kg': masse, 'allong_m': allong, 'u_allong': u_allong})
donnees

In [ ]:
def droite(x, a, b):
    return a * x + b

popt, pcov = optimize.curve_fit(droite, donnees['m_kg'], donnees['allong_m'],
                                sigma=donnees['u_allong'], absolute_sigma=True)
a, b = popt
u_a, u_b = np.sqrt(np.diag(pcov))

print("pente   a =", formate(a, u_a, "m/kg"))
print("ordonnée b =", formate(b, u_b, "m"))

# raideur du ressort : k = g / a
g = 9.81
k = g / a
u_k = k * u_a / a
print("raideur k =", formate(k, u_k, "N/m"))

## Qualité de l'ajustement : le $\chi^2$ réduit

Le $\chi^2$ mesure l'écart entre les données et le modèle, **en unités d'incertitudes** :

$$\chi^2 = \sum_i \left(\frac{y_i - f(x_i)}{u_i}\right)^2$$

Le **$\chi^2$ réduit** $\chi^2_\nu = \chi^2/\nu$, avec $\nu = N - (\text{nombre de paramètres})$ degrés de liberté, doit valoir **environ 1** si le modèle est bon et les incertitudes bien estimées. Beaucoup plus grand que 1 : modèle inadapté ou incertitudes sous-estimées. Beaucoup plus petit que 1 : incertitudes surestimées.

In [ ]:
residus = donnees['allong_m'] - droite(donnees['m_kg'], a, b)
chi2 = np.sum((residus / donnees['u_allong'])**2)
nu = len(donnees) - 2
print(f"chi2 = {chi2:.2f}, nu = {nu}, chi2 réduit = {chi2/nu:.2f}")
print(f"p-value = {stats.chi2.sf(chi2, nu):.2f}")

Une *p-value* raisonnable (par exemple entre 0,05 et 0,95) indique que l'ajustement est compatible avec les données.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(6, 6),
                               gridspec_kw={'height_ratios': [3, 1]})
xx = np.linspace(0, 0.42, 100)
ax1.errorbar(donnees['m_kg'], donnees['allong_m'] * 100, yerr=donnees['u_allong'] * 100,
             fmt='o', capsize=3, label="mesures")
ax1.plot(xx, droite(xx, a, b) * 100, label="ajustement")
ax1.set_ylabel("allongement (cm)"); ax1.legend()

ax2.errorbar(donnees['m_kg'], residus / donnees['u_allong'], yerr=1, fmt='o', capsize=3)
ax2.axhline(0, color='k', lw=0.8)
ax2.set_ylabel("résidus normalisés")
ax2.set_xlabel("masse (kg)")
plt.show()

Les **résidus normalisés** doivent se répartir de façon aléatoire autour de 0, avec la plupart des points dans $[-2, 2]$. Une tendance (courbure, dérive) signale un modèle inadapté.


> **Exercice : et si l'incertitude est sous-estimée ?**
>
> Refaites l'ajustement ci-dessus en supposant (à tort) une incertitude de $0{,}0005$ m au lieu de $0{,}002$ m. Que devient le $\chi^2$ réduit ? Que devient l'incertitude sur la raideur ? Comment interprétez-vous ces résultats ?

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
sig = 0.0005
popt2, pcov2 = optimize.curve_fit(droite, donnees['m_kg'], donnees['allong_m'],
                                  sigma=sig * np.ones(len(donnees)), absolute_sigma=True)
res2 = donnees['allong_m'] - droite(donnees['m_kg'], *popt2)
chi2_2 = np.sum((res2 / sig)**2)
print("chi2 réduit =", round(chi2_2 / nu, 1))
print("u(k) =", (g / popt2[0]) * np.sqrt(pcov2[0, 0]) / popt2[0])
# chi2 réduit ~ 18 >> 1 : les résidus sont 4 fois plus grands que les incertitudes annoncées.
# L'incertitude sur k paraît plus petite, mais elle est fausse : incertitudes sous-estimées.

# Bilan

| Situation | Outil Python |
| --------- | ------------ |
| plusieurs mesures d'une même grandeur | `x.mean()`, `x.std(ddof=1)`, `s/np.sqrt(N)` |
| intervalle de confiance (petit $N$) | `stats.t.interval(...)` |
| instrument (une lecture) | $u = a/\sqrt{3}$ |
| combiner des sources indépendantes | `np.hypot(u1, u2)` ou $\sqrt{u_1^2 + u_2^2}$ |
| propager à une formule | SymPy (`diff`) ou Monte-Carlo (`rng.normal`) |
| comparer à une référence | écart normalisé $E_n$ |
| ajuster un modèle | `curve_fit(..., sigma=u, absolute_sigma=True)` |
| qualité de l'ajustement | $\chi^2$ réduit, résidus normalisés |


> **Exercice de synthèse : mesure de la célérité du son**
>
> On mesure le temps $t$ mis par un son pour parcourir différentes distances $d$ (tube de Kundt, ou claquement et écho) :
>
> | d (m) | 20 | 40 | 60 | 80 | 100 |
> | ----- | -- | -- | -- | -- | --- |
> | t (ms) | 59 | 118 | 172 | 233 | 291 |
>
> On prend $u(t) = 3$ ms pour chaque mesure ; la distance est supposée connue avec une incertitude négligeable.
>
> 1. Ajustez $d = c\, t$ (droite passant par l'origine : ajustez seulement la pente).
> 2. Donnez $c$ avec son incertitude-type.
> 3. Calculez le $\chi^2$ réduit et l'écart normalisé avec la valeur de référence $343$ m/s.

In [ ]:
# Votre code ici

In [ ]:
#@title Solution { display-mode: "form" }
d = np.array([20, 40, 60, 80, 100.0])
t = np.array([59, 118, 172, 233, 291]) * 1e-3
u_t = 3e-3

# d = c * t  <=>  on ajuste t = d / c = p * d, avec p = 1/c
modele = lambda d, p: p * d
(p,), pcov = optimize.curve_fit(modele, d, t, sigma=u_t * np.ones_like(t), absolute_sigma=True)
u_p = np.sqrt(pcov[0, 0])
c, u_c = 1 / p, u_p / p**2
print("c =", formate(c, u_c, "m/s"))

chi2 = np.sum(((t - modele(d, p)) / u_t)**2)
print("chi2 réduit =", round(chi2 / (len(d) - 1), 2))
print("E_n =", round(abs(c - 343) / u_c, 2))

---

*Fin du carnet 8.*